In [1]:
import sys; sys.path.append('../..')
from src.modelos.dados import ler_base, preparar, dividir
from src.modelos.variaveis import criar_variaveis
from src.modelos.treino import _para_modelo, metricas, metricas_por_grupo, calibracao_por_faixa, salvar
from src.modelos.xgb import GRADE, buscar_parametros, treinar_final

base = preparar(ler_base())
treino, teste = dividir(base)
treino = criar_variaveis(treino)
teste = criar_variaveis(teste, referencia=treino)

# 1. Escolher parâmetros (valida em 2015)
grade = buscar_parametros(treino)
print(grade.head(5))

# 2. Modelo final calibrado em 2015
from src.modelos.xgb import treinar_final_completo

melhor = grade.iloc[0]
params = {k: int(melhor[k]) for k in GRADE}
m = treinar_final_completo(treino, params, int(melhor['n_arvores']))

# 3. Teste 2016 (uma vez só)
p = m.predict_proba(_para_modelo(teste))[:, 1]
print('\nTeste:', metricas(teste['default'], p))
print(metricas_por_grupo(teste, p, 'pais'))
print(calibracao_por_faixa(teste['default'], p))

salvar(m, 'xgboost_v2', {'params': params, 'teste': metricas(teste['default'], p)})

c:\Users\samue\OneDrive\Desktop\tcs\Auditor-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   max_depth  min_child_weight  reg_lambda  n_arvores  contratos  \
0          3                20           1        231       5943   
1          4                20           1        160       5943   
2          3                 5           1        282       5943   
3          3                 5          10        268       5943   
4          4                20          10        161       5943   

   default_real  default_previsto     auc      ks   brier  
0        0.6655            0.6655  0.7222  0.3356  0.1896  
1        0.6655            0.6658  0.7218  0.3353  0.1900  
2        0.6655            0.6619  0.7214  0.3312  0.1902  
3        0.6655            0.6658  0.7213  0.3328  0.1900  
4        0.6655            0.6666  0.7212  0.3356  0.1900  

Teste: {'contratos': 7311, 'default_real': 0.6708, 'default_previsto': 0.5656, 'auc': 0.7271, 'ks': 0.3366, 'brier': 0.1987}
      contratos  default_real  default_previsto     auc      ks   brier
pais                             

WindowsPath('C:/Users/samue/OneDrive/Desktop/tcs/Auditor-Project/modelos_salvos/xgboost_v2.joblib')